# 12 · Travel-time friction surface 2019 — Malaria Atlas Project (India window)

**Source:** Malaria Atlas Project (MAP), *Global friction surfaces 2019* (motorised and walking-only), released
2020 with Weiss, D.J. et al. (2020) Global maps of travel time to healthcare facilities. *Nature Medicine* 26,
1835–1838. **Licence:** CC BY 4.0. Served by MAP's GeoServer (`data.malariaatlas.org`, WCS 2.0.1).

**What the values are:** for every ~1 km pixel, the time in **minutes per metre** to cross it by the fastest
means available there — roads (OSM and other road data, by class), railways, navigable water, otherwise
off-road travel by land cover, slowed by slope (SRTM) and blocked by large water bodies, plus border
crossings. So travel time = sum of (minutes per metre × metres) along the fastest path.

**Files** (`data/raw/friction/`): motorised and walking friction, and — for checking our results — MAP's own
2019 travel time to its healthcare facilities. Only the India window is requested; a file already saved is
not requested again.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import os
if "postgis" in os.environ.get("PROJ_LIB", "").lower():  # an old PROJ database from a PostGIS install breaks rasterio
    os.environ.pop("PROJ_LIB")
import shutil

import numpy as np
import rasterio
import requests

OUT_DIR = PROJECT / "data" / "raw" / "friction"
WCS = ("https://data.malariaatlas.org/geoserver/Accessibility/ows?service=WCS&version=2.0.1&request=GetCoverage"
       "&coverageId={cov}&subset=Long({w},{e})&subset=Lat({s},{n})&format=image/geotiff")
INDIA = (68.0, 97.5, 6.5, 37.5)  # west, east, south, north (includes Andaman & Nicobar and Lakshadweep)
COVERAGES = {
    "motorized_friction_2019": "Accessibility__202001_Global_Motorized_Friction_Surface",
    "walking_friction_2019": "Accessibility__202001_Global_Walking_Only_Friction_Surface",
    # MAP's own travel-time maps, used only to check our travel times
    "motorized_travel_time_to_healthcare_2019": "Accessibility__202001_Global_Motorized_Travel_Time_to_Healthcare",
    "walking_travel_time_to_healthcare_2019": "Accessibility__202001_Global_Walking_Only_Travel_Time_To_Healthcare",
}
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

### `get_india()`

In [3]:
def get_india(name, cov):
    """Ask MAP's GeoServer (WCS 2.0.1) for the India window of one global layer; save it once."""
    f = OUT_DIR / f"india_{name}.tif"
    if f.exists():
        print("already saved:", f.name)
        return f
    w, e, s, n = INDIA
    r = requests.get(WCS.format(cov=cov, w=w, e=e, s=s, n=n), headers=HEADERS, timeout=900)
    if not r.ok or "tiff" not in r.headers.get("content-type", ""):
        raise SystemExit(f"{name}: server answered {r.status_code} {r.headers.get('content-type')}: {r.text[:300]}")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    f.write_bytes(r.content)
    print(f"saved: {f.name} ({len(r.content) / 1e6:.0f} MB)")
    return f

## Run

In [4]:
for name, cov in COVERAGES.items():
    get_india(name, cov)

saved: india_motorized_friction_2019.tif (59 MB)


saved: india_walking_friction_2019.tif (59 MB)


saved: india_motorized_travel_time_to_healthcare_2019.tif (59 MB)


saved: india_walking_travel_time_to_healthcare_2019.tif (59 MB)


## Check

Grid size and the implied speeds (60 / (minutes per metre × 1000) = km/h).

In [5]:
for name in COVERAGES:
    with rasterio.open(OUT_DIR / f"india_{name}.tif") as r:
        a = r.read(1, masked=True)
        print(f"{name}: {r.width} x {r.height}, {r.res[0] * 3600:.0f} arc-seconds, valid {a.count():,} cells")
        if "friction" in name:  # minutes per metre -> km/h = 60 / (value * 1000)
            q = np.percentile(a.compressed(), [5, 50, 95])
            print(f"   minutes per metre 5/50/95 %: {q.round(5)} -> speed {(60 / (q * 1000)).round(1)} km/h")
        else:
            print(f"   minutes to nearest MAP healthcare facility, median {float(np.ma.median(a)):.0f}")

motorized_friction_2019: 3540 x 3720, 30 arc-seconds, valid 13,168,800 cells
   minutes per metre 5/50/95 %: [0.0015  0.00316 0.07128] -> speed [40.  19.   0.8] km/h


walking_friction_2019: 3540 x 3720, 30 arc-seconds, valid 13,168,800 cells
   minutes per metre 5/50/95 %: [0.012   0.03694 0.07129] -> speed [5.  1.6 0.8] km/h


motorized_travel_time_to_healthcare_2019: 3540 x 3720, 30 arc-seconds, valid 8,513,239 cells


   minutes to nearest MAP healthcare facility, median 58
walking_travel_time_to_healthcare_2019: 3540 x 3720, 30 arc-seconds, valid 8,513,239 cells


   minutes to nearest MAP healthcare facility, median 276
